In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/Bank_Loan_Data.csv")
print(f"Raw shape: {df.shape}")

Raw shape: (38576, 24)


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38576 entries, 0 to 38575
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     38576 non-null  int64  
 1   address_state          38576 non-null  object 
 2   application_type       38576 non-null  object 
 3   emp_length             38576 non-null  object 
 4   emp_title              37138 non-null  object 
 5   grade                  38576 non-null  object 
 6   home_ownership         38576 non-null  object 
 7   issue_date             38576 non-null  object 
 8   last_credit_pull_date  38576 non-null  object 
 9   last_payment_date      38576 non-null  object 
 10  loan_status            38576 non-null  object 
 11  next_payment_date      38576 non-null  object 
 12  member_id              38576 non-null  int64  
 13  purpose                38576 non-null  object 
 14  sub_grade              38576 non-null  object 
 15  te

In [9]:
df.head()

,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,sub_grade,term,verification_status,annual_income,dti,installment,int_rate,loan_amount,total_acc,total_payment
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,11-02-2021,13-09-2021,13-04-2021,...,C4,60 months,Source Verified,30000.0,0.0100,59.83,0.1527,2500,4,1009
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,01-01-2021,14-12-2021,15-01-2021,...,E1,36 months,Source Verified,48000.0,0.0535,109.43,0.1864,3000,4,3939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,05-01-2021,12-12-2021,09-01-2021,...,C5,36 months,Not Verified,50000.0,0.2088,421.65,0.1596,12000,11,3522
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,25-02-2021,12-12-2021,12-03-2021,...,B2,60 months,Source Verified,42000.0,0.0540,97.06,0.1065,4500,9,4911
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,01-01-2021,14-12-2021,15-01-2021,...,A1,36 months,Verified,83000.0,0.0231,106.53,0.0603,3500,28,3835


In [3]:
# Deduplication
df.drop_duplicates(subset=["id"], inplace=True)
print(f"After dedup: {df.shape}")

After dedup: (38576, 24)


In [11]:
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

df["term"] = df["term"].str.strip()

cat_cols = ["grade", "sub_grade", "home_ownership", "purpose",
            "verification_status", "application_type", "loan_status"]
df[cat_cols] = df[cat_cols].apply(lambda c: c.str.lower().str.strip())

In [12]:
# Date formatting
date_cols = ["issue_date", "last_credit_pull_date",
             "last_payment_date", "next_payment_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%d-%m-%Y", errors="coerce")

df["issue_year"]  = df["issue_date"].dt.year
df["issue_month"] = df["issue_date"].dt.month

# Days since last payment
report_date = df["last_payment_date"].max()
df["days_since_last_payment"] = (report_date - df["last_payment_date"]).dt.days

In [13]:
df["int_rate_pct"] = (df["int_rate"] * 100).round(2)

df["term_months"] = df["term"].str.extract(r"(\d+)").astype(int)

# Clipping outliers in annual_income
INC_CAP = 500_000
df["annual_income_capped"] = df["annual_income"].clip(upper=INC_CAP)

# DTI check — already 0–0.30 range, no action needed; flag if > 0.43 (threshold)
df["dti_flag"] = (df["dti"] > 0.43).astype(int)

In [14]:
# Encoding Employee period
emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
    "4 years": 4,  "5 years": 5, "6 years": 6, "7 years": 7,
    "8 years": 8,  "9 years": 9, "10+ years": 10
}
df["emp_length_num"] = df["emp_length"].map(emp_map)

In [15]:
# Checking for null values
df.isna().sum()

,0
id,0
address_state,0
application_type,0
emp_length,0
emp_title,1438
grade,0
home_ownership,0
issue_date,0
last_credit_pull_date,0
last_payment_date,0


In [16]:
df["emp_title"] = df["emp_title"].fillna("unknown")

# Verify no remaining nulls in critical columns
critical = ["loan_amount", "annual_income", "dti", "int_rate",
            "grade", "loan_status", "term_months"]
assert df[critical].isnull().sum().sum() == 0, "Nulls in critical columns!"

In [17]:
# Debt-to-Income Ratio bucket
def dti_bucket(d):
    if d < 0.10:   return "low"
    elif d < 0.20: return "moderate"
    elif d < 0.30: return "high"
    else:          return "very_high"

df["dti_bucket"] = df["dti"].apply(dti_bucket)

In [18]:
# Loan-to-Income Ratio
df["lti_ratio"] = (df["loan_amount"] / df["annual_income_capped"]).round(4)

In [19]:
# Repayment Rate
df["expected_total"] = df["installment"] * df["term_months"]
df["repayment_rate"] = (df["total_payment"] / df["expected_total"]).clip(0, 1).round(4)

In [20]:
# Binary default flag (Charged Off = 1, Fully Paid / Current = 0)
df["is_default"] = (df["loan_status"] == "charged off").astype(int)

In [21]:
# Grade numeric score (A=7 best → G=1 worst) for correlation analysis
grade_score = {"a": 7, "b": 6, "c": 5, "d": 4, "e": 3, "f": 2, "g": 1}
df["grade_score"] = df["grade"].map(grade_score)

In [22]:
# Rule based Scoring

def compute_risk_score(row):
    score = 0

    # DTI (max 25)
    score += min(row["dti"] / 0.43, 1) * 25

    # Grade (max 30): G=30, A≈4.3
    score += (8 - row["grade_score"]) * (30 / 7)

    # LTI (max 20)
    score += min(row["lti_ratio"] / 0.5, 1) * 20

    # Term
    if row["term_months"] == 60:
        score += 15

    # Verification
    if row["verification_status"] == "not verified":
        score += 10

    return round(min(score, 100), 2)

df["risk_score"] = df.apply(compute_risk_score, axis=1)

In [23]:
def risk_tier(s):
    if s < 30:   return "low_risk"
    elif s < 55: return "medium_risk"
    elif s < 75: return "high_risk"
    else:        return "very_high_risk"

df["risk_tier"] = df["risk_score"].apply(risk_tier)

print("\nRisk tier distribution:")
print(df["risk_tier"].value_counts())
print("\nDefault rate by risk tier:")
print(df.groupby("risk_tier")["is_default"].mean().round(3))


Risk tier distribution:
risk_tier
medium_risk       19024
low_risk          16076
high_risk          3448
very_high_risk       28
Name: count, dtype: int64

Default rate by risk tier:
risk_tier
high_risk         0.284
low_risk          0.079
medium_risk       0.162
very_high_risk    0.321
Name: is_default, dtype: float64


In [24]:
output_cols = [
    # IDs
    "id", "member_id",
    # Loan attributes
    "loan_amount", "term_months", "int_rate_pct", "installment",
    "grade", "sub_grade", "grade_score", "purpose",
    # Applicant attributes
    "annual_income_capped", "dti", "dti_bucket", "dti_flag",
    "lti_ratio", "emp_length_num", "home_ownership",
    "verification_status", "address_state", "application_type",
    # Dates
    "issue_date", "issue_year", "issue_month", "days_since_last_payment",
    # KPIs
    "total_payment", "expected_total", "repayment_rate",
    "total_acc",
    # Target & scoring
    "loan_status", "is_default", "risk_score", "risk_tier",
]

In [25]:
# Exporting cleaned dataset
df_clean = df[output_cols].copy()
df_clean.to_csv("bank_loan_clean.csv", index=False)
print(f"\nCleaned dataset saved: {df_clean.shape[0]} rows × {df_clean.shape[1]} cols")


Cleaned dataset saved: 38576 rows × 32 cols
